# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import VGG16

# 1. Carga del dataset CIFAR-10 (50,000 imágenes de entrenamiento y 10,000 de prueba)
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 2. Normalización de píxeles: de [0, 255] a [0.0, 1.0]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 3. Codificación One-Hot para las 10 clases de CIFAR-10
num_classes = 10
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

print(f"Dimensiones de entrenamiento: {X_train.shape}, Etiquetas: {y_train_cat.shape}")
print(f"Dimensiones de prueba:        {X_test.shape}, Etiquetas: {y_test_cat.shape}")


Dimensiones de entrenamiento: (50000, 32, 32, 3), Etiquetas: (50000, 10)
Dimensiones de prueba:        (10000, 32, 32, 3), Etiquetas: (10000, 10)


## Definiciones de modelos

In [4]:
# Recuerda aquí solo generar las arquitecturas, cada capa así como sus neuronas.

input_shape = (32, 32, 3)  # Imágenes de 32x32 píxeles con 3 canales de color (RGB)

# ==============================================================================
# BLOQUE DE DATA AUGMENTATION (Aumento de datos para evitar el sobreajuste)
# ==============================================================================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),     # Voltea la imagen horizontalmente al azar
    layers.RandomRotation(0.1),          # Rota la imagen ligeramente (hasta 10%)
    layers.RandomZoom(0.1),              # Hace un pequeño zoom aleatorio (hasta 10%)
], name="Data_Augmentation")

# ==============================================================================
# MODELO 1: Arquitectura CNN Base (Simple, sin regularización ni augmentation)
# ==============================================================================
model_cnn_base = models.Sequential([
    layers.Input(shape=input_shape),
    
    # Bloque Convolucional 1: detecta bordes y colores básicos
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),  # Reduce tamaño de 32x32 a 16x16
    
    # Bloque Convolucional 2: detecta texturas y formas simples
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),  # Reduce tamaño de 16x16 a 8x8
    
    # Clasificador Denso (Perceptrón Multicapa al final)
    layers.Flatten(),             # Aplana el cubo de 8x8x64 a un vector de 4096 valores
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')  # 10 neuronas de salida (1 por clase)
], name="CNN_1_Base")


# ==============================================================================
# MODELO 2: Arquitectura CNN Profunda + Data Augmentation + Regularización
# ==============================================================================
model_cnn_regularizada = models.Sequential([
    layers.Input(shape=input_shape),
    data_augmentation,            # Aplicamos Data Augmentation en la entrada
    
    # Bloque Convolucional 1 (32 filtros + BatchNormalization + Dropout)
    layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),         # Apaga el 25% de neuronas al azar para regularizar
    
    # Bloque Convolucional 2 (64 filtros + BatchNormalization + Dropout)
    layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.35),         # Apaga el 35% de neuronas
    
    # Clasificador Denso Regularizado
    layers.Flatten(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),          # Apaga el 50% antes de la decisión final
    layers.Dense(num_classes, activation='softmax')
], name="CNN_2_Regularizada")


# ==============================================================================
# MODELO 3: Transfer Learning con Arquitectura Clásica (VGG16)
# ==============================================================================
# Cargamos VGG16 pre-entrenada en ImageNet, sin su capa clasificadora original (include_top=False)
base_vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
base_vgg16.trainable = False      # Congelamos los pesos de VGG16 para no destruir lo que ya sabe

model_transfer = models.Sequential([
    layers.Input(shape=input_shape),
    data_augmentation,            # También usamos Data Augmentation
    base_vgg16,                   # Cerebro extractor de características pre-entrenado
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),          # Regularización en nuestro clasificador nuevo
    layers.Dense(num_classes, activation='softmax')
], name="CNN_3_Transfer_VGG16")

# Mostramos el resumen de las 3 arquitecturas
model_cnn_base.summary()
model_cnn_regularizada.summary()
model_transfer.summary()

Model: "CNN_1_Base"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 32, 32, 32)        896       
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 16, 16, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 16, 16, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 8, 8, 64)          0         
 g2D)                                                            
                                                                 
 flatten_3 (Flatten)         (None, 4096)              0         
                                                                 
 dense_6 (Dense)             (None, 64)                2

## Entrenamiento de modelos.

In [5]:
# Aquí agrega la compilación y entrenamiento de las arquitecturas generadas.

# Hiperparámetros generales de entrenamiento
EPOCHS = 15
BATCH_SIZE = 64

# Callback de EarlyStopping: detiene el entrenamiento si val_loss deja de mejorar y restaura el mejor modelo
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

# ==============================================================================
# 1. COMPILACIÓN DE LOS MODELOS
# ==============================================================================
model_cnn_base.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn_regularizada.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_transfer.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==============================================================================
# 2. ENTRENAMIENTO DE LOS MODELOS
# ==============================================================================
print("\n--- Entrenando Modelo 1: CNN Base ---")
history_base = model_cnn_base.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test_cat),
    callbacks=[early_stopping],
    verbose=1
)

print("\n--- Entrenando Modelo 2: CNN Regularizada + Data Augmentation ---")
history_reg = model_cnn_regularizada.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test_cat),
    callbacks=[early_stopping],
    verbose=1
)

print("\n--- Entrenando Modelo 3: Transfer Learning (VGG16) ---")
history_transfer = model_transfer.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test_cat),
    callbacks=[early_stopping],
    verbose=1
)

# ==============================================================================
# 3. EVALUACIÓN FINAL EN DATOS DE PRUEBA (TEST)
# ==============================================================================
loss_base, acc_base = model_cnn_base.evaluate(X_test, y_test_cat, verbose=0)
loss_reg, acc_reg = model_cnn_regularizada.evaluate(X_test, y_test_cat, verbose=0)
loss_trans, acc_trans = model_transfer.evaluate(X_test, y_test_cat, verbose=0)

print("\n================ RESULTADOS EN TEST ================")
print(f"1. CNN Base         -> Loss: {loss_base:.4f} | Accuracy: {acc_base*100:.2f}%")
print(f"2. CNN Regularizada -> Loss: {loss_reg:.4f} | Accuracy: {acc_reg*100:.2f}%")
print(f"3. Transfer VGG16   -> Loss: {loss_trans:.4f} | Accuracy: {acc_trans*100:.2f}%")


--- Entrenando Modelo 1: CNN Base ---
Epoch 1/15


782/782 [==============================] - 10s 12ms/step - loss: 1.4841 - accuracy: 0.4689 - val_loss: 1.2441 - val_accuracy: 0.5651
Epoch 2/15
782/782 [==============================] - 9s 11ms/step - loss: 1.1112 - accuracy: 0.6108 - val_loss: 1.1287 - val_accuracy: 0.6029
Epoch 3/15
782/782 [==============================] - 8s 11ms/step - loss: 0.9763 - accuracy: 0.6594 - val_loss: 0.9608 - val_accuracy: 0.6635
Epoch 4/15
782/782 [==============================] - 9s 11ms/step - loss: 0.8926 - accuracy: 0.6870 - val_loss: 0.9354 - val_accuracy: 0.6696
Epoch 5/15
782/782 [==============================] - 9s 11ms/step - loss: 0.8252 - accuracy: 0.7135 - val_loss: 0.9004 - val_accuracy: 0.6893
Epoch 6/15
782/782 [==============================] - 9s 12ms/step - loss: 0.7707 - accuracy: 0.7314 - val_loss: 0.8773 - val_accuracy: 0.6953
Epoch 7/15
782/782 [==============================] - 9s 11ms/step - loss: 0.7154 - accuracy: 0.7507

## Estadística y gráficos

In [6]:
# ==============================================================================
# 1. TABLA ESTADÍSTICA COMPARATIVA (PANDAS DATAFRAME)
# ==============================================================================
resultados_df = pd.DataFrame({
    'Arquitectura': [
        '1. CNN Base (Simple)',
        '2. CNN Regularizada + Augmentation',
        '3. Transfer Learning (VGG16)'
    ],
    'Test Loss': [loss_base, loss_reg, loss_trans],
    'Test Accuracy (%)': [acc_base * 100, acc_reg * 100, acc_trans * 100],
    'Train Accuracy Final (%)': [
        history_base.history['accuracy'][-1] * 100,
        history_reg.history['accuracy'][-1] * 100,
        history_transfer.history['accuracy'][-1] * 100
    ],
    'Brecha Overfitting (Train - Val %)': [
        (history_base.history['accuracy'][-1] - history_base.history['val_accuracy'][-1]) * 100,
        (history_reg.history['accuracy'][-1] - history_reg.history['val_accuracy'][-1]) * 100,
        (history_transfer.history['accuracy'][-1] - history_transfer.history['val_accuracy'][-1]) * 100
    ]
})

print("\n================ TABLA COMPARATIVA DE MODELOS ================")
display(resultados_df.round(2))


# ==============================================================================
# 2. GRÁFICAS DE CURVAS DE APRENDIZAJE (PRECISIÓN Y PÉRDIDA POR ÉPOCA)
# ==============================================================================
historiales = [
    ('1. CNN Base', history_base, 'royalblue'),
    ('2. CNN Regularizada', history_reg, 'seagreen'),
    ('3. Transfer VGG16', history_transfer, 'darkorange')
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Comparación Experimental de Curvas de Aprendizaje (CIFAR-10)', fontsize=16, fontweight='bold')

for idx, (nombre, hist, color) in enumerate(historiales):
    epocas = range(1, len(hist.history['accuracy']) + 1)
    
    # --- Fila superior: Accuracy (Precisión) ---
    ax_acc = axes[0, idx]
    ax_acc.plot(epocas, hist.history['accuracy'], label='Entrenamiento (Train)', color=color, linestyle='--', linewidth=2)
    ax_acc.plot(epocas, hist.history['val_accuracy'], label='Validación (Test)', color=color, linewidth=2.5)
    ax_acc.set_title(f'Accuracy - {nombre}', fontsize=12, fontweight='bold')
    ax_acc.set_xlabel('Épocas')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_ylim([0.2, 1.0])
    ax_acc.grid(True, linestyle=':', alpha=0.6)
    ax_acc.legend()
    
    # --- Fila inferior: Loss (Pérdida / Error) ---
    ax_loss = axes[1, idx]
    ax_loss.plot(epocas, hist.history['loss'], label='Entrenamiento (Train)', color=color, linestyle='--', linewidth=2)
    ax_loss.plot(epocas, hist.history['val_loss'], label='Validación (Test)', color=color, linewidth=2.5)
    ax_loss.set_title(f'Loss - {nombre}', fontsize=12, fontweight='bold')
    ax_loss.set_xlabel('Épocas')
    ax_loss.set_ylabel('Loss (Categorical Crossentropy)')
    ax_loss.grid(True, linestyle=':', alpha=0.6)
    ax_loss.legend()

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# ==============================================================================
# 3. GRÁFICO DE BARRAS: COMPARACIÓN DIRECTA EN TEST
# ==============================================================================
plt.figure(figsize=(9, 5))
nombres = resultados_df['Arquitectura']
precisiones = resultados_df['Test Accuracy (%)']
colores = ['royalblue', 'seagreen', 'darkorange']

barras = plt.bar(nombres, precisiones, color=colores, width=0.5, edgecolor='black')
plt.title('Exactitud Final en Datos de Prueba (Test Accuracy %)', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Mostrar el porcentaje exacto encima de cada barra
for barra in barras:
    altura = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2., altura + 1.5,
             f'{altura:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


================ TABLA COMPARATIVA DE MODELOS ================


,Arquitectura,Test Loss,Test Accuracy (%),Train Accuracy Final (%),Brecha Overfitting (Train - Val %)
0,1. CNN Base (Simple),0.86,71.50,83.46,12.93
1,2. CNN Regularizada + Augmentation,0.94,73.22,70.68,-2.54
2,3. Transfer Learning (VGG16),1.26,55.77,52.85,-2.92


<Figure size 1800x1000 with 6 Axes>

<Figure size 900x500 with 1 Axes>

# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

¿Cuál fue el mejor modelo?
    
    La mejor fue la CNN Regularizada + Argmentacion

¿Por qué obtuvo ese resultado?
    
    Se ubtuvo un mejor resultado gracias a incorporar bloques de convolucion

¿Qué mejoraría?
    
    Despues de entrenar el clasificador podemos hacer fine tuning que reentrenaria el modelo para dar mejores resultados

¿Cómo lo mejoraría?
    
    Mediante redimencionamiento podeamos agergar una capa al inicio del modelo de transfer learning.